# Global Topic Renaming — iGEM Teams (Part 2)

The per-cluster naming in Part 1 works locally: each topic is named in
isolation. This can produce **duplicate or ambiguous names** when two clusters
cover related sub-themes.

This notebook addresses that by giving the LLM a **global view** of all
**iGEM Teams** topics at once. We use **OpenAI function calling** so the
model returns a structured array of `(topic_id, name)` pairs — one per cluster —
and asks for distinct names; the notebook warns if a name is missing or repeated.

Overwrites `teams_topic_names.txt`, adding a `global_name` column.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 03-topic_names/, where the aux/ package
# and setup_run.py reside; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import OPENAI_MODEL
from setup_run import setup
from aux.openai_client import load_prompts, make_client
from aux.tables import load_topic_names, save_topic_names
from aux.global_rename import rename_topics_global

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
PREFIX = "teams"
MODEL  = OPENAI_MODEL
RUN    = setup(corpus=PREFIX)  # today's run folder: assets/<date>/03/

prompts = load_prompts()
client = make_client()

## 1. Load Part 1 results

In [3]:
names = load_topic_names(RUN, PREFIX)
print(f"Teams: {len(names)} topics")
names[["topic", "name", "description"]].head()

Teams: 154 topics


,topic,name,description
0,0,Synthetic Biology for Controlled Gene Expressi...,This cluster focuses on harnessing synthetic b...
1,1,Pest and Vector Control,This cluster focuses on harnessing advanced ge...
2,2,Synthetic Biology for Plant Disease Detection ...,This cluster focuses on leveraging synthetic b...
3,3,Biofilm and Quorum Sensing Control,This cluster focuses on leveraging synthetic b...
4,4,Synthetic Biology Design and Automation Tools,This cluster centers on the advancement of too...


## 2. Global rename (function calling)

In [4]:
renamed = rename_topics_global(names, client, prompts, model=MODEL)
renamed[["topic", "name", "global_name", "description"]]

,topic,name,global_name,description
0,0,Synthetic Biology for Controlled Gene Expressi...,Gene Expression Patterning,This cluster focuses on harnessing synthetic b...
1,1,Pest and Vector Control,Pest and Vector Genetic Control,This cluster focuses on harnessing advanced ge...
2,2,Synthetic Biology for Plant Disease Detection ...,Plant Disease Synthetic Diagnostics,This cluster focuses on leveraging synthetic b...
3,3,Biofilm and Quorum Sensing Control,Biofilm and Quorum Disruption,This cluster focuses on leveraging synthetic b...
4,4,Synthetic Biology Design and Automation Tools,Synthetic Biology Design Automation,This cluster centers on the advancement of too...
...,...,...,...,...
149,149,Engineered Microbial Therapeutics,Engineered Microbial Therapeutics,This cluster centers on harnessing Synthetic B...
150,150,Synthetic Biology for Bacterial Detection and ...,Bacterial Detection and Control,This cluster focuses on leveraging synthetic b...
151,151,Synthetic Biology for Mental Health and Gut Mi...,Gut Microbiome and Mental Health,This cluster focuses on harnessing synthetic b...
152,152,Synthetic Biology for Environmental Bioremedia...,Water Pollution Bioremediation,This cluster centers on leveraging synthetic b...


## 3. Save final results

In [5]:
save_topic_names(RUN, renamed, PREFIX)
print(f"Saved → {RUN.dir / f'{PREFIX}_topic_names.txt'} (added global_name)")

Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/topic_models/teams_topic_names.txt (added global_name)
